In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)

df = sns.load_dataset("titanic")

leaky = ['class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
df = df.drop(columns=leaky)

X = df.drop(columns=['survived'])
y = df['survived']

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=15/85, stratify=y_tmp, random_state=42)

num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so  = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)
X_train_t = preprocess.transform(X_train)
X_val_t   = preprocess.transform(X_val)
X_test_t  = preprocess.transform(X_test)

In [9]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# LOGISTIC REGRESSION
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_t, y_train)
y_pred_log = log_model.predict(X_test_t)

# LINEAR REGRESSION
lin_model = LinearRegression()
lin_model.fit(X_train_t, y_train)
y_pred_lin_cont = lin_model.predict(X_test_t)
y_pred_lin = np.where(y_pred_lin_cont >= 0.5, 1, 0)

def evaluate_classification(y_true, y_pred, model_name):
    print(f"{model_name}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score:  {f1_score(y_true, y_pred):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate_classification(y_test, y_pred_log, "LOGISTIC REGRESSION")
evaluate_classification(y_test, y_pred_lin, "LINEAR REGRESSION (Threshold = 0.5)")

LOGISTIC REGRESSION
Accuracy:  0.7761
Precision: 0.7442
Recall:    0.6275
F1-score:  0.6809
Confusion Matrix:
[[72 11]
 [19 32]]

LINEAR REGRESSION (Threshold = 0.5)
Accuracy:  0.7612
Precision: 0.7021
Recall:    0.6471
F1-score:  0.6735
Confusion Matrix:
[[69 14]
 [18 33]]



Đánh giá và so sánh kết quả của mô hình Logistic Regression: So với Linear regression, các chỉ số accuracy, precision và F1-score của mô hình Logistic regression đều cao hơn nhưng chỉ số recall lại thấp hơn. Điều này cho thấy Logistic regression phù hợp hơn cho bài toán này, mặc dù Linear regression tìm ra nhiều người sống sót hơn (do recall cao hơn) nhưng nó lại dự đoán số lượng người chết thành người sống cao hơn (do precision thấp hơn). Nên trong bài toán phân loại nhị phân thì nên dùng Logistic regression hơn.